Pure LSTM RNN Autoencoder–Binary–Classifier

Complete three-head multi-task workflow based on recurrent LSTM processing:
1. LSTM encoder-decoder autoencoder reconstruction head
2. LSTM binary safe/fault head
3. LSTM seven-class state/fault classifier

The data loading, session-wise cross-validation, safe-only reconstruction training,
state/PWM/load-conditioned thresholds, decision fusion, exports, and manual testing
remain identical to the MLP and Conv1D versions so their results can be compared fairly.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib tensorflow tf2onnx onnx onnxruntime

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedShuffleSplit,
)
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_PATH = (
    r"../../data/Motor_data_model"
)

SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s", "PWM", "Irms", "Vrms", "Current_Variance",
    "Current_Crest_Factor", "Vib_RMS", "Vib_Min", "Vib_Q1",
    "Vib_Median", "Vib_Q3", "Vib_Max", "Vib_Variance", "Vib_Skewness",
]

FINAL_LABELS = [
    "motor_off", "turning_off", "turning_on", "normal",
    "electrical_fault", "mechanical_fault", "thermal_fault",
]

LABEL_TO_ID = {label: index for index, label in enumerate(FINAL_LABELS)}

SAFE_LABELS = ["motor_off", "turning_off", "turning_on", "normal"]
FAULT_LABELS = ["electrical_fault", "mechanical_fault", "thermal_fault"]

SAFE_CLASS_IDS = np.asarray([LABEL_TO_ID[l] for l in SAFE_LABELS], dtype=np.int64)
FAULT_CLASS_IDS = np.asarray([LABEL_TO_ID[l] for l in FAULT_LABELS], dtype=np.int64)

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)

WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
BATCH_SIZE = 128
MAX_EPOCHS = 180

LEARNING_RATE = 1e-3

# CHANGED: RECON_LOSS_WEIGHT raised from 0.20 -> 1.00. Now that the
# reconstruction encoder is fully isolated (CELL 8 dual-encoder fix),
# it no longer competes with anomaly/class gradients for a shared
# LSTM trunk, so a low weight is no longer needed to protect those
# heads — and it was only slowing the recon branch's convergence.
RECON_LOSS_WEIGHT = 1.00
ANOMALY_LOSS_WEIGHT = 0.80
CLASS_LOSS_WEIGHT = 1.00

#early warning parameter
EARLY_WARNING_CONSECUTIVE_WINDOWS = 3

# Compact pure-LSTM recurrent architecture.
SHARED_LSTM_UNITS = 6
AE_LATENT_LSTM_UNITS = 4
AE_DECODER_LSTM_UNITS = 6
ANOMALY_LSTM_UNITS = 1
CLASS_LSTM_UNITS = N_CLASSES
LSTM_DROPOUT = 0.0
LSTM_RECURRENT_DROPOUT = 0.0
MODEL_SIZE_TARGET_KB = 50.0

CLASS_CONF_THRESHOLD = 0.65
ANOMALY_PROB_THRESHOLD = 0.65
RECON_QUANTILE = 0.995

# ADDED: reconstruction severity tiers (matches DANN/CNN notebooks).
# Pure post-processing on reconstruction_mse / local_baseline — no
# retraining involved.
SEVERITY_MILD_MULTIPLIER = 2.0
SEVERITY_MODERATE_MULTIPLIER = 4.0
SEVERITY_SEVERE_MULTIPLIER = 8.0

# NOTE: the 1.01 upper edge already acts as a soft ceiling for PWM
# values slightly above 100% caused by ulong-scaling logging
# artifacts at full duty cycle. The real fix is clipping PWM at the
# source (see CELL 4) so those rows don't silently contaminate the
# "75_100" bin's calibration.
PWM_BIN_EDGES = np.asarray([
    0.00, 0.20, 0.35, 0.45, 0.55, 0.65, 0.75, 1.01,
], dtype=np.float32)

PWM_BIN_NAMES = [
    "0_20", "20_35", "35_45", "45_55", "55_65", "65_75", "75_100",
]

LOAD_PROXY_FEATURE = "Irms"
LOAD_SPLIT_QUANTILE = 0.50
MIN_THRESHOLD_WINDOWS = 30

LOAD_STATE_NAMES = ["low_load", "high_load"]

OUTPUT_DIR = "lstm_rnn_ae_3head_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Model: Pure LSTM RNN 3-Head Hybrid (dual-encoder)")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Classes:", FINAL_LABELS)
print("Safe classes:", SAFE_LABELS)
print("Fault classes:", FAULT_LABELS)

In [ ]:
# ==========================================================
# CELL 3 — deterministic file-to-class mapping
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if any(term in path_text for term in [
        "motor_being_off",
        "turning_off",
        "transition_off",
        "off_spikes",
    ]):
        return "turning_off"

    if any(term in path_text for term in [
        "motor_being_on",
        "turning_on",
        "transition_on",
        "on_spikes",
    ]):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if any(term in path_text for term in [
        "electrical fault",
        "electrical_fault",
        "abnormal_current",
    ]):
        return "electrical_fault"

    if any(term in path_text for term in [
        "mechanical fault",
        "mechanical_fault",
        "abnormal_vibration",
    ]):
        return "mechanical_fault"

    if any(term in path_text for term in [
        "thermal fault",
        "thermal_fault",
        "abnormal_temp",
    ]):
        return "thermal_fault"

    if (
        "/normal/" in path_text
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal"

    return None


def anomaly_target_from_class_id(class_id):
    return int(
        class_id in FAULT_CLASS_IDS
    )

# ----------------------------------------------------------
# CHANGED: reconstruction reference = NORMAL WITH LOAD only
# (same rule as the DANN notebook). Without-load normal,
# motor_off and transitions are NOT used to train the AE.
# ----------------------------------------------------------
def is_normal_with_load_file(path, label):
    if label != "normal":
        return False

    path_text = normalized_path(path)

    # Check without-load first because it also contains the word "load".
    without_load_terms = [
        "without_load",
        "without load",
        "no_load",
        "no load",
        "unloaded",
        "normal_noload",
        "normal_no_load",
    ]

    with_load_terms = [
        "with_load",
        "with load",
        "loaded",
        "normal_load",
    ]

    if any(term in path_text for term in without_load_terms):
        return False

    if any(term in path_text for term in with_load_terms):
        return True

    raise ValueError(
        "Cannot identify loaded/unloaded normal file: "
        f"{path}"
    )



In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    missing_columns = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in {path.name}: {missing_columns}")

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    clean = clean / SCALE_DIVISOR

    # ADDED: PWM duty cycle is physically bounded to [0, 1]. Values
    # recorded slightly above 1.0 are ulong-scaling logging artifacts
    # at the 100% boundary, not a real operating condition. Without
    # this clip they silently contaminate the top PWM bin's
    # reconstruction-threshold calibration.
    if "PWM" in clean.columns:
        overshoot_count = int((clean["PWM"] > 1.0).sum())
        if overshoot_count > 0:
            print(f"  Clipping {overshoot_count} PWM rows > 1.0 in {path.name}")
        clean["PWM"] = clean["PWM"].clip(upper=1.0)

    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(f"{path.name} has only {len(clean)} valid rows.")

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

sessions = []

for session_id, file_path in enumerate(files):
    label = label_for(file_path)

    if label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    class_id = LABEL_TO_ID[label]

    sessions.append({
        "session_id": session_id,
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": class_id,
        "anomaly": anomaly_target_from_class_id(class_id),
        "reconstruction_reference": int(is_normal_with_load_file(file_path, label)),  # CHANGED
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError(
        "No labeled sessions were loaded."
    )

class_counts = pd.Series([
    session["label"]
    for session in sessions
]).value_counts().reindex(
    FINAL_LABELS,
    fill_value=0,
)

print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [
    label
    for label in FINAL_LABELS
    if class_counts[label] == 0
]

if missing_classes:
    raise RuntimeError(
        f"No files found for classes: {missing_classes}"
    )


# ----------------------------------------------------------
# CHANGED: session_id -> 1 if normal-with-load (AE reference)
# ----------------------------------------------------------
RECON_REFERENCE_BY_SESSION = {
    session["session_id"]: session["reconstruction_reference"]
    for session in sessions
}

print("\nNORMAL WITH LOAD — USED TO TRAIN THE RECONSTRUCTION AE")
for session in sessions:
    if session["reconstruction_reference"] == 1:
        print("  ", session["file"])

print("\nNORMAL WITHOUT LOAD — NOT USED FOR AE TRAINING")
for session in sessions:
    if session["label"] == "normal" and session["reconstruction_reference"] == 0:
        print("  ", session["file"])

if not any(RECON_REFERENCE_BY_SESSION.values()):
    raise RuntimeError("No normal-with-load files were identified.")


In [ ]:
# ==========================================================
# CELL 6 — create 30 × 14 windows
# ==========================================================
def make_windows_from_array(X_raw):
    starts = np.arange(
        0,
        len(X_raw) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_dataset_from_sessions(selected_sessions):
    all_windows = []
    all_class_labels = []
    all_anomaly_labels = []
    all_files = []
    all_starts = []
    all_groups = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"]
        )

        if windows is None:
            continue

        number_of_windows = len(windows)

        all_windows.append(windows)

        all_class_labels.extend(
            [session["class_id"]] * number_of_windows
        )

        all_anomaly_labels.extend(
            [session["anomaly"]] * number_of_windows
        )

        all_files.extend(
            [session["file"]] * number_of_windows
        )

        all_starts.extend(
            starts.tolist()
        )

        all_groups.extend(
            [session["session_id"]] * number_of_windows
        )

    if not all_windows:
        raise RuntimeError(
            "No windows were created."
        )

    return (
        np.concatenate(
            all_windows,
            axis=0,
        ).astype(np.float32),
        np.asarray(
            all_class_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_anomaly_labels,
            dtype=np.float32,
        ),
        np.asarray(
            all_files,
            dtype=object,
        ),
        np.asarray(
            all_starts,
            dtype=np.int64,
        ),
        np.asarray(
            all_groups,
            dtype=np.int64,
        ),
    )

In [ ]:
# ==========================================================
# CELL 7 — normalization utilities
# ==========================================================
def calculate_normalization_statistics(X_train):
    flattened_rows = X_train.reshape(
        -1,
        N_FEATURES,
    )

    mean = np.mean(
        flattened_rows,
        axis=0,
    ).astype(np.float32)

    variance = np.var(
        flattened_rows,
        axis=0,
    ).astype(np.float32)

    variance = np.maximum(
        variance,
        1e-12,
    )

    return mean, variance


def normalize_windows_numpy(
    X,
    mean,
    variance,
):
    return (
        (
            X
            - mean.reshape(
                1,
                1,
                N_FEATURES,
            )
        )
        / np.sqrt(
            variance.reshape(
                1,
                1,
                N_FEATURES,
            )
            + 1e-7
        )
    ).astype(np.float32)

In [ ]:
# ==========================================================
# CELL 8 — LSTM RNN 3-head model — DUAL-ENCODER VERSION
#
# WHY THIS CHANGE:
# Previously a single `shared_sequence` (shared_lstm) fed BOTH the
# reconstruction encoder (ae_encoder_lstm) AND the anomaly/class
# heads. anomaly_output/class_output train with full weight on
# every window, while reconstruction trains only on safe windows —
# so the classifier pulls the shared LSTM state toward
# fault-discriminative structure, which then gets decoded,
# corrupting reconstruction error as an off-manifold signal. Same
# failure mode already fixed in the DANN and CNN notebooks; this
# brings the LSTM notebook into structural alignment for a fair
# comparison.
#
# FIX: two fully independent LSTM stacks, same three named outputs:
#   recon_shared_lstm -> ae_encoder_lstm -> repeat -> decoder LSTM
#       -> reconstruction_output
#       (gradients ONLY from window_mse_loss on safe windows)
#   shared_lstm -> anomaly_lstm / class_lstm -> anomaly_output /
#       class_output
#       (gradients from anomaly + class losses on all windows)
#
# Still pure-LSTM: no Dense/Conv1D/GRU/SimpleRNN/TimeDistributed
# layers anywhere, so the existing forbidden-layer check still
# passes unchanged.
# ==========================================================
def window_mse_loss(y_true, y_pred):
    """Return one reconstruction MSE value per complete window."""
    squared_error = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_error, axis=[1, 2])


def build_hybrid_model(
    normalization_mean,
    normalization_variance,
):
    input_layer = tf.keras.Input(
        shape=(WINDOW_LEN, N_FEATURES),
        name="motor_window",
    )

    normalizer = tf.keras.layers.Normalization(
        axis=-1,
        mean=normalization_mean,
        variance=normalization_variance,
        name="input_normalization",
    )
    normalized = normalizer(input_layer)

    # ------------------------------------------------------
    # BRANCH A — dedicated reconstruction LSTM stack (isolated).
    # Only reconstruction_output's loss ever touches these layers.
    # ------------------------------------------------------
    recon_sequence = tf.keras.layers.LSTM(
        SHARED_LSTM_UNITS,
        return_sequences=True,
        activation="tanh",
        recurrent_activation="sigmoid",
        use_bias=True,
        dropout=LSTM_DROPOUT,
        recurrent_dropout=LSTM_RECURRENT_DROPOUT,
        unroll=True,
        name="recon_shared_lstm",
    )(normalized)

    ae_latent = tf.keras.layers.LSTM(
        AE_LATENT_LSTM_UNITS,
        return_sequences=False,
        activation="tanh",
        recurrent_activation="sigmoid",
        use_bias=True,
        dropout=LSTM_DROPOUT,
        recurrent_dropout=LSTM_RECURRENT_DROPOUT,
        unroll=True,
        name="ae_encoder_lstm",
    )(recon_sequence)

    repeated_latent = tf.keras.layers.RepeatVector(
        WINDOW_LEN,
        name="ae_repeat_latent",
    )(ae_latent)

    reconstruction_output = tf.keras.layers.LSTM(
        N_FEATURES,
        return_sequences=True,
        activation="linear",
        recurrent_activation="sigmoid",
        use_bias=True,
        dropout=0.0,
        recurrent_dropout=0.0,
        unroll=True,
        name="reconstruction_output",
    )(repeated_latent)

    # ------------------------------------------------------
    # BRANCH B — shared discriminative LSTM stack (anomaly +
    # class only). Completely separate weights from Branch A.
    # ------------------------------------------------------
    shared_sequence = tf.keras.layers.LSTM(
        SHARED_LSTM_UNITS,
        return_sequences=True,
        activation="tanh",
        recurrent_activation="sigmoid",
        use_bias=True,
        dropout=LSTM_DROPOUT,
        recurrent_dropout=LSTM_RECURRENT_DROPOUT,
        unroll=True,
        name="shared_lstm",
    )(normalized)

    anomaly_logit = tf.keras.layers.LSTM(
        ANOMALY_LSTM_UNITS,
        return_sequences=False,
        activation="linear",
        recurrent_activation="sigmoid",
        use_bias=True,
        dropout=LSTM_DROPOUT,
        recurrent_dropout=LSTM_RECURRENT_DROPOUT,
        unroll=True,
        name="anomaly_lstm",
    )(shared_sequence)

    anomaly_output = tf.keras.layers.Activation(
        "sigmoid", name="anomaly_output",
    )(anomaly_logit)

    class_logits = tf.keras.layers.LSTM(
        CLASS_LSTM_UNITS,
        return_sequences=False,
        activation="linear",
        recurrent_activation="sigmoid",
        use_bias=True,
        dropout=LSTM_DROPOUT,
        recurrent_dropout=LSTM_RECURRENT_DROPOUT,
        unroll=True,
        name="class_lstm",
    )(shared_sequence)

    class_output = tf.keras.layers.Activation(
        "softmax", name="class_output",
    )(class_logits)

    model = tf.keras.Model(
        inputs=input_layer,
        outputs={
            "reconstruction_output": reconstruction_output,
            "anomaly_output": anomaly_output,
            "class_output": class_output,
        },
        name="dual_encoder_pure_lstm_3head",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={
            "reconstruction_output": window_mse_loss,
            "anomaly_output": tf.keras.losses.BinaryCrossentropy(),
            "class_output": tf.keras.losses.SparseCategoricalCrossentropy(),
        },
        loss_weights={
            "reconstruction_output": RECON_LOSS_WEIGHT,
            "anomaly_output": ANOMALY_LOSS_WEIGHT,
            "class_output": CLASS_LOSS_WEIGHT,
        },
        metrics={
            "anomaly_output": [
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.AUC(name="auc"),
            ],
            "class_output": [
                tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            ],
        },
    )

    return model


# Show architecture and estimated weight sizes only once.
_preview_model = build_hybrid_model(
    normalization_mean=np.zeros(N_FEATURES, dtype=np.float32),
    normalization_variance=np.ones(N_FEATURES, dtype=np.float32),
)

_preview_model.summary()

_forbidden_trainable_types = (
    tf.keras.layers.Dense,
    tf.keras.layers.Conv1D,
    tf.keras.layers.GRU,
    tf.keras.layers.SimpleRNN,
    tf.keras.layers.TimeDistributed,
)

_forbidden_layers = [
    layer.name for layer in _preview_model.layers
    if isinstance(layer, _forbidden_trainable_types)
]

if _forbidden_layers:
    raise RuntimeError(
        "The model is not pure LSTM. Forbidden layers found: "
        + ", ".join(_forbidden_layers)
    )

_lstm_layers = [
    layer for layer in _preview_model.layers
    if isinstance(layer, tf.keras.layers.LSTM)
]

print("LSTM layers:", [layer.name for layer in _lstm_layers])
print("No Dense/CNN/GRU/SimpleRNN/TimeDistributed layers found.")

_preview_parameters = _preview_model.count_params()

print("Total parameters:", _preview_parameters)
print("Estimated Float32 weight size KB:", round(_preview_parameters * 4 / 1024, 2))
print("Estimated INT8 weight size KB:", round(_preview_parameters / 1024, 2))

del _preview_model

In [ ]:
# ==========================================================
# CELL 9 — training weights and dataset split helpers
# ==========================================================
def calculate_class_sample_weights(y_class):
    present_classes = np.unique(y_class)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_class,
    )

    class_weight_lookup = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            present_classes,
            class_weights,
        )
    }

    return np.asarray([
        class_weight_lookup[int(class_id)]
        for class_id in y_class
    ], dtype=np.float32)


def calculate_anomaly_sample_weights(y_anomaly):
    y_integer = y_anomaly.astype(np.int64)
    present_classes = np.unique(y_integer)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_integer,
    )

    lookup = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            present_classes,
            weights,
        )
    }

    return np.asarray([
        lookup[int(value)]
        for value in y_integer
    ], dtype=np.float32)


def recon_reference_for_groups(groups):
    """CHANGED: 1.0 for windows from normal-with-load files, else 0.0."""
    return np.asarray(
        [RECON_REFERENCE_BY_SESSION[int(g)] for g in groups],
        dtype=np.float32,
    )


def make_training_sample_weights(
    y_class,
    y_anomaly,
    groups,
):
    # CHANGED: AE trains ONLY on normal-with-load windows
    reconstruction_weights = recon_reference_for_groups(groups)

    class_weights = calculate_class_sample_weights(
        y_class
    )

    anomaly_weights = calculate_anomaly_sample_weights(
        y_anomaly
    )

    return {
        "reconstruction_output":
            reconstruction_weights,
        "anomaly_output":
            anomaly_weights,
        "class_output":
            class_weights,
    }


def split_train_validation_by_group(
    X,
    y_class,
    y_anomaly,
    files,
    starts,
    groups,
):
    """
    Create validation data at SESSION level, never by overlapping windows.
    This avoids leakage and does not require StratifiedGroupShuffleSplit,
    which is unavailable in several scikit-learn versions.
    """
    unique_groups, first_indices = np.unique(
        groups,
        return_index=True,
    )
    group_labels = y_class[first_indices]

    class_group_counts = pd.Series(
        group_labels
    ).value_counts()

    if class_group_counts.min() < 2:
        raise RuntimeError(
            "Each class needs at least 2 sessions inside the training fold "
            "to create a stratified session-wise validation split. "
            f"Current session counts: {class_group_counts.to_dict()}"
        )

    validation_fraction = max(
        0.15,
        N_CLASSES / len(unique_groups),
    )
    validation_fraction = min(
        validation_fraction,
        0.40,
    )

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=validation_fraction,
        random_state=SEED,
    )

    group_train_pos, group_validation_pos = next(
        splitter.split(
            unique_groups,
            group_labels,
        )
    )

    train_group_ids = unique_groups[group_train_pos]
    validation_group_ids = unique_groups[group_validation_pos]

    train_index = np.flatnonzero(
        np.isin(groups, train_group_ids)
    )
    validation_index = np.flatnonzero(
        np.isin(groups, validation_group_ids)
    )

    if np.intersect1d(
        groups[train_index],
        groups[validation_index],
    ).size:
        raise RuntimeError(
            "Session leakage detected between training and validation."
        )

    train_data = (
        X[train_index],
        y_class[train_index],
        y_anomaly[train_index],
        files[train_index],
        starts[train_index],
        groups[train_index],
    )

    validation_data = (
        X[validation_index],
        y_class[validation_index],
        y_anomaly[validation_index],
        files[validation_index],
        starts[validation_index],
        groups[validation_index],
    )

    return train_data, validation_data

In [ ]:
# ==========================================================
# CELL 11 — fit one fold/model
# ==========================================================
def fit_model_with_group_validation(
    X_train_full,
    y_class_full,
    y_anomaly_full,
    files_full,
    starts_full,
    groups_full,
):
    (
        train_data,
        validation_data,
    ) = split_train_validation_by_group(
        X_train_full,
        y_class_full,
        y_anomaly_full,
        files_full,
        starts_full,
        groups_full,
    )

    (
        X_train,
        y_class_train,
        y_anomaly_train,
        _,
        _,
        groups_train,
    ) = train_data

    (
        X_validation,
        y_class_validation,
        y_anomaly_validation,
        _,
        _,
        groups_validation,
    ) = validation_data

    normal_train_mask = (
    y_class_train == LABEL_TO_ID["normal"]
)

    normalization_mean, normalization_variance = (
        calculate_normalization_statistics(
            X_train[normal_train_mask]
        )
    )

    X_train_normalized = normalize_windows_numpy(
        X_train,
        normalization_mean,
        normalization_variance,
    )

    X_validation_normalized = normalize_windows_numpy(
        X_validation,
        normalization_mean,
        normalization_variance,
    )

    model = build_hybrid_model(
        normalization_mean,
        normalization_variance,
    )

    sample_weights = make_training_sample_weights(
        y_class_train,
        y_anomaly_train,
        groups_train,
    )

    validation_sample_weights = {
        "reconstruction_output":
            recon_reference_for_groups(groups_validation),  # CHANGED
            "anomaly_output":
            np.ones(
                len(y_anomaly_validation),
                dtype=np.float32,
            ),
        "class_output":
            np.ones(
                len(y_class_validation),
                dtype=np.float32,
            ),
    }

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_class_output_accuracy",
            mode="max",
            patience=20,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_class_output_loss",
            mode="min",
            factor=0.5,
            patience=7,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_train,
        {
            "reconstruction_output":
                X_train_normalized,
            "anomaly_output":
                y_anomaly_train,
            "class_output":
                y_class_train,
        },
        sample_weight=sample_weights,
        validation_data=(
            X_validation,
            {
                "reconstruction_output":
                    X_validation_normalized,
                "anomaly_output":
                    y_anomaly_validation,
                "class_output":
                    y_class_validation,
            },
            validation_sample_weights,
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    return (
        model,
        normalization_mean,
        normalization_variance,
        history,
    )

In [ ]:
# ==========================================================
# CELL 12 — decision fusion and conditional thresholds
# (CHANGED: added severity_level_and_action() and
# get_local_normal_baseline(), matching DANN/CNN notebooks.
# fuse_decisions now also returns severity_ratio/severity_label/
# recommended_action per window.)
# ==========================================================
def get_pwm_bin_index(mean_pwm):
    bin_index = np.searchsorted(PWM_BIN_EDGES, mean_pwm, side="right") - 1
    return int(np.clip(bin_index, 0, len(PWM_BIN_NAMES) - 1))


def get_conditional_reconstruction_threshold(
    predicted_class_id, mean_pwm, mean_irms,
):
    predicted_label = FINAL_LABELS[int(predicted_class_id)]

    if predicted_label in ["motor_off", "turning_off", "turning_on"]:
        return float(
            STATE_RECON_THRESHOLDS.get(predicted_label, RECON_THRESHOLD)
        )

    bin_index = get_pwm_bin_index(mean_pwm)
    bin_name = PWM_BIN_NAMES[bin_index]

    load_split = float(
        PWM_LOAD_SPLITS.get(bin_name, GLOBAL_NORMAL_LOAD_SPLIT)
    )

    load_state = "high_load" if mean_irms >= load_split else "low_load"

    return float(
        NORMAL_PWM_LOAD_RECON_THRESHOLDS.get(bin_name, {}).get(
            load_state,
            NORMAL_PWM_RECON_THRESHOLDS.get(bin_name, RECON_THRESHOLD),
        )
    )


def get_local_normal_baseline(mean_pwm):
    """ADDED — per-bin median reconstruction MSE for severity scoring."""
    bin_index = get_pwm_bin_index(mean_pwm)
    bin_name = PWM_BIN_NAMES[bin_index]

    return float(
        LOCAL_NORMAL_BASELINES.get(bin_name, GLOBAL_NORMAL_BASELINE)
    )


def severity_level_and_action(severity_ratio):
    """ADDED — matches DANN/CNN severity tiering exactly."""
    if severity_ratio < SEVERITY_MILD_MULTIPLIER:
        return "normal", "No action needed."
    elif severity_ratio < SEVERITY_MODERATE_MULTIPLIER:
        return "mild", "Log and monitor; no immediate action."
    elif severity_ratio < SEVERITY_SEVERE_MULTIPLIER:
        return "moderate", "Schedule inspection soon."
    else:
        return "severe", "Stop motor / inspect immediately."


def fuse_decisions(
    class_probabilities,
    anomaly_probabilities,
    reconstruction_mse,
    reconstruction_thresholds,
    mean_pwm,
):
    class_ids = np.argmax(class_probabilities, axis=1).astype(np.int64)
    class_confidence = np.max(class_probabilities, axis=1)

    reconstruction_anomaly_flags = reconstruction_mse > reconstruction_thresholds
    binary_anomaly_flags = anomaly_probabilities >= ANOMALY_PROB_THRESHOLD

    confident_known_fault_flags = (
        np.isin(class_ids, FAULT_CLASS_IDS)
        & (class_confidence >= CLASS_CONF_THRESHOLD)
    )

    anomaly_flags = (
        binary_anomaly_flags
        | reconstruction_anomaly_flags
        | confident_known_fault_flags
    )

    # ADDED: severity ratio + tier, informational for every window.
    local_baselines = np.asarray(
        [get_local_normal_baseline(pwm) for pwm in mean_pwm], dtype=np.float32,
    )
    safe_baselines = np.where(local_baselines > 0, local_baselines, 1.0)
    severity_ratios = reconstruction_mse / safe_baselines

    severity_labels = []
    severity_actions = []
    for ratio in severity_ratios:
        label, action = severity_level_and_action(float(ratio))
        severity_labels.append(label)
        severity_actions.append(action)

    final_labels = []

    for (
        class_id, confidence, binary_flag,
        reconstruction_flag, known_fault_flag,
    ) in zip(
        class_ids, class_confidence, binary_anomaly_flags,
        reconstruction_anomaly_flags, confident_known_fault_flags,
    ):
        predicted_label = FINAL_LABELS[int(class_id)]

        if known_fault_flag:
            final_labels.append(predicted_label)
        elif binary_flag or reconstruction_flag:
            final_labels.append("unknown_fault")
        elif class_id in SAFE_CLASS_IDS and confidence >= CLASS_CONF_THRESHOLD:
            final_labels.append(predicted_label)
        else:
            final_labels.append("uncertain_safe")

    return (
        class_ids,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        anomaly_flags,
        np.asarray(final_labels, dtype=object),
        local_baselines,
        severity_ratios.astype(np.float32),
        np.asarray(severity_labels, dtype=object),
        np.asarray(severity_actions, dtype=object),
    )

In [ ]:
# ==========================================================
# CELL 13 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [
        session["class_id"]
        for session in sessions
    ],
    "group": [
        session["session_id"]
        for session in sessions
    ],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
all_true = []
all_pred = []
all_anomaly_true = []
all_anomaly_pred = []
all_reconstruction_mse = []
all_mean_pwm = []
all_mean_irms = []
all_window_predictions = []
cv_best_epochs = []

for fold, (
    train_index,
    test_index,
) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print("\n" + "=" * 70)
    print("FOLD", fold)
    print("=" * 70)

    train_sessions = [
        sessions[index]
        for index in train_index
    ]

    test_sessions = [
        sessions[index]
        for index in test_index
    ]

    (
        X_train_full,
        y_class_train_full,
        y_anomaly_train_full,
        train_files,
        train_starts,
        train_groups,
    ) = build_dataset_from_sessions(
        train_sessions
    )

    (
        X_test,
        y_class_test,
        y_anomaly_test,
        test_files,
        test_starts,
        _,
    ) = build_dataset_from_sessions(
        test_sessions
    )

    (
        model,
        normalization_mean,
        normalization_variance,
        history,
    ) = fit_model_with_group_validation(
        X_train_full,
        y_class_train_full,
        y_anomaly_train_full,
        train_files,
        train_starts,
        train_groups,
    )

    validation_accuracy_history = history.history.get(
        "val_class_output_accuracy",
        [],
    )
    if validation_accuracy_history:
        best_epoch = int(
            np.argmax(validation_accuracy_history) + 1
        )
    else:
        best_epoch = len(history.history.get("loss", []))

    cv_best_epochs.append(best_epoch)
    print("Best validation epoch:", best_epoch)

    X_test_normalized = normalize_windows_numpy(
        X_test,
        normalization_mean,
        normalization_variance,
    )

    prediction_dict = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    reconstructed = prediction_dict[
        "reconstruction_output"
    ]

    anomaly_probabilities = prediction_dict[
        "anomaly_output"
    ].reshape(-1)

    class_probabilities = prediction_dict[
        "class_output"
    ]

    class_predictions = np.argmax(
        class_probabilities,
        axis=1,
    ).astype(np.int64)

    anomaly_predictions = (
        anomaly_probabilities
        >= ANOMALY_PROB_THRESHOLD
    ).astype(np.int64)

    reconstruction_mse = np.mean(
        np.square(
            reconstructed
            - X_test_normalized
        ),
        axis=(1, 2),
    )

    pwm_feature_index = FEATURE_COLUMNS.index(
        "PWM"
    )
    irms_feature_index = FEATURE_COLUMNS.index(
        LOAD_PROXY_FEATURE
    )

    test_window_mean_pwm = np.mean(
        X_test[:, :, pwm_feature_index],
        axis=1,
    ).astype(np.float32)

    test_window_mean_irms = np.mean(
        X_test[:, :, irms_feature_index],
        axis=1,
    ).astype(np.float32)
    

    all_true.extend(
        y_class_test.tolist()
    )

    all_pred.extend(
        class_predictions.tolist()
    )

    all_anomaly_true.extend(
        y_anomaly_test.astype(np.int64).tolist()
    )

    all_anomaly_pred.extend(
        anomaly_predictions.tolist()
    )

    all_reconstruction_mse.extend(
        reconstruction_mse.tolist()
    )

    all_mean_pwm.extend(
        test_window_mean_pwm.tolist()
    )

    all_mean_irms.extend(
        test_window_mean_irms.tolist()
    )

    fold_metrics = {
        "fold": fold,
        "accuracy":
            accuracy_score(
                y_class_test,
                class_predictions,
            ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_class_test,
                class_predictions,
            ),
        "macro_precision":
            precision_score(
                y_class_test,
                class_predictions,
                average="macro",
                zero_division=0,
            ),
        "macro_recall":
            recall_score(
                y_class_test,
                class_predictions,
                average="macro",
                zero_division=0,
            ),
        "macro_f1":
            f1_score(
                y_class_test,
                class_predictions,
                average="macro",
                zero_division=0,
            ),
        "anomaly_accuracy":
            accuracy_score(
                y_anomaly_test,
                anomaly_predictions,
            ),
        "anomaly_f1":
            f1_score(
                y_anomaly_test,
                anomaly_predictions,
                zero_division=0,
            ),
        "mean_reconstruction_mse":
            float(
                np.mean(
                    reconstruction_mse
                )
            ),
    }

    cv_metrics.append(
        fold_metrics
    )

    fold_predictions = pd.DataFrame({
        "fold": fold,
        "file": test_files,
        "start_row": test_starts,
        "mean_pwm": test_window_mean_pwm,
        "mean_irms": test_window_mean_irms,
        "true_class_id":
            y_class_test,
        "true_label": [
            FINAL_LABELS[index]
            for index in y_class_test
        ],
        "pred_class_id":
            class_predictions,
        "pred_label": [
            FINAL_LABELS[index]
            for index in class_predictions
        ],
        "class_confidence":
            np.max(
                class_probabilities,
                axis=1,
            ),
        "true_anomaly":
            y_anomaly_test.astype(np.int64),
        "anomaly_probability":
            anomaly_probabilities,
        "anomaly_prediction":
            anomaly_predictions,
        "reconstruction_mse":
            reconstruction_mse,
    })

    for class_id, label in enumerate(
        FINAL_LABELS
    ):
        fold_predictions[
            f"probability_{label}"
        ] = class_probabilities[
            :,
            class_id,
        ]

    all_window_predictions.append(
        fold_predictions
    )

    print(
        pd.Series(
            fold_metrics
        )
    )

    print("\nFOLD CLASSIFICATION REPORT")
    print(
        classification_report(
            y_class_test,
            class_predictions,
            labels=np.arange(N_CLASSES),
            target_names=FINAL_LABELS,
            zero_division=0,
        )
    )

    fold_cm = confusion_matrix(
        y_class_test,
        class_predictions,
        labels=np.arange(N_CLASSES),
    )

    pd.DataFrame(
        fold_cm,
        index=FINAL_LABELS,
        columns=FINAL_LABELS,
    ).to_csv(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_confusion_matrix.csv",
        )
    )

    fig, axis = plt.subplots(figsize=(9, 8))
    ConfusionMatrixDisplay(
        fold_cm,
        display_labels=FINAL_LABELS,
    ).plot(
        ax=axis,
        values_format="d",
        xticks_rotation=45,
        colorbar=False,
    )
    axis.set_title(f"Fold {fold} — Session-wise Test Confusion Matrix")
    fig.tight_layout()
    fig.savefig(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_confusion_matrix.png",
        ),
        dpi=180,
    )
    plt.show()
    plt.close(fig)

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_training_history.csv",
        ),
        index=False,
    )

    tf.keras.backend.clear_session()


In [ ]:
# ==========================================================
# CELL 14 — combined CV report and reconstruction threshold
# (CHANGED: added GLOBAL_NORMAL_BASELINE and LOCAL_NORMAL_BASELINES,
# needed by the severity scoring added in CELL 12.)
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_predictions_df = pd.concat(all_window_predictions, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_predictions_df.to_csv(os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False)

all_true = np.asarray(all_true, dtype=np.int64)
all_pred = np.asarray(all_pred, dtype=np.int64)
all_anomaly_true = np.asarray(all_anomaly_true, dtype=np.int64)
all_anomaly_pred = np.asarray(all_anomaly_pred, dtype=np.int64)
all_mean_pwm = np.asarray(all_mean_pwm, dtype=np.float32)
all_mean_irms = np.asarray(all_mean_irms, dtype=np.float32)
all_reconstruction_mse = np.asarray(all_reconstruction_mse, dtype=np.float32)

NORMAL_CLASS_ID = LABEL_TO_ID["normal"]

normal_mask = all_true == NORMAL_CLASS_ID

if not np.any(normal_mask):
    raise RuntimeError(
        "No held-out normal-running windows are available "
        "for reconstruction-threshold calibration."
    )

normal_reconstruction_mse = all_reconstruction_mse[normal_mask]

RECON_THRESHOLD = float(
    np.quantile(normal_reconstruction_mse, RECON_QUANTILE)
)

STATE_RECON_THRESHOLDS = {}

for state_label in ["motor_off", "turning_off", "turning_on"]:
    state_class_id = LABEL_TO_ID[state_label]
    state_errors = all_reconstruction_mse[all_true == state_class_id]

    if len(state_errors) >= MIN_THRESHOLD_WINDOWS:
        state_threshold = float(np.quantile(state_errors, RECON_QUANTILE))
    else:
        state_threshold = RECON_THRESHOLD
        print(
            f"WARNING: state {state_label} has only {len(state_errors)} "
            "held-out windows. Using global safe fallback threshold."
        )

    STATE_RECON_THRESHOLDS[state_label] = state_threshold
    print(
        f"State {state_label} | windows={len(state_errors)} | "
        f"threshold={state_threshold:.8f}"
    )



if not np.any(normal_mask):
    raise RuntimeError(
        "No held-out steady normal windows are available for PWM/load threshold calibration."
    )


normal_mean_pwm = all_mean_pwm[normal_mask]
normal_mean_irms = all_mean_irms[normal_mask]

GLOBAL_NORMAL_LOAD_SPLIT = float(np.quantile(normal_mean_irms, LOAD_SPLIT_QUANTILE))
GLOBAL_NORMAL_THRESHOLD = float(np.quantile(normal_reconstruction_mse, RECON_QUANTILE))

# ADDED: global baseline for severity scoring fallback.
GLOBAL_NORMAL_BASELINE = float(np.median(normal_reconstruction_mse))

PWM_LOAD_SPLITS = {}
NORMAL_PWM_RECON_THRESHOLDS = {}
NORMAL_PWM_LOAD_RECON_THRESHOLDS = {}
# ADDED
LOCAL_NORMAL_BASELINES = {}
threshold_rows = []

for bin_index, bin_name in enumerate(PWM_BIN_NAMES):
    lower_pwm = PWM_BIN_EDGES[bin_index]
    upper_pwm = PWM_BIN_EDGES[bin_index + 1]

    pwm_mask = (normal_mean_pwm >= lower_pwm) & (normal_mean_pwm < upper_pwm)
    pwm_errors = normal_reconstruction_mse[pwm_mask]
    pwm_irms = normal_mean_irms[pwm_mask]

    if len(pwm_errors) >= MIN_THRESHOLD_WINDOWS:
        pwm_threshold = float(np.quantile(pwm_errors, RECON_QUANTILE))
        pwm_baseline = float(np.median(pwm_errors))
        load_split = float(np.quantile(pwm_irms, LOAD_SPLIT_QUANTILE))
    else:
        pwm_threshold = GLOBAL_NORMAL_THRESHOLD
        pwm_baseline = GLOBAL_NORMAL_BASELINE
        load_split = GLOBAL_NORMAL_LOAD_SPLIT
        print(
            f"WARNING: PWM bin {bin_name} has only {len(pwm_errors)} "
            "normal windows. Using global normal fallback values."
        )

    PWM_LOAD_SPLITS[bin_name] = load_split
    NORMAL_PWM_RECON_THRESHOLDS[bin_name] = pwm_threshold
    LOCAL_NORMAL_BASELINES[bin_name] = pwm_baseline
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name] = {}

    for load_state in LOAD_STATE_NAMES:
        if load_state == "low_load":
            load_mask = pwm_mask & (normal_mean_irms < load_split)
        else:
            load_mask = pwm_mask & (normal_mean_irms >= load_split)

        load_errors = normal_reconstruction_mse[load_mask]

        if len(load_errors) >= MIN_THRESHOLD_WINDOWS:
            load_threshold = float(np.quantile(load_errors, RECON_QUANTILE))
        else:
            load_threshold = pwm_threshold
            print(
                f"WARNING: {bin_name}/{load_state} has only {len(load_errors)} "
                "normal windows. Using its PWM-bin threshold."
            )

        NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name][load_state] = load_threshold

        threshold_rows.append({
            "operating_state": "normal",
            "pwm_bin": bin_name,
            "lower_pwm": float(lower_pwm),
            "upper_pwm": float(upper_pwm),
            "load_state": load_state,
            "irms_split": float(load_split),
            "normal_windows": int(len(load_errors)),
            "reconstruction_threshold": float(load_threshold),
            "local_baseline": float(pwm_baseline),
        })

        print(
            f"Normal {bin_name}/{load_state} | windows={len(load_errors)} | "
            f"Irms split={load_split:.8f} | threshold={load_threshold:.8f}"
        )

threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(
    os.path.join(CV_DIR, "conditional_reconstruction_thresholds.csv"), index=False,
)

print("\nGlobal safe reconstruction fallback:", RECON_THRESHOLD)

overall_metrics = {
    "accuracy": accuracy_score(all_true, all_pred),
    "balanced_accuracy": balanced_accuracy_score(all_true, all_pred),
    "macro_precision": precision_score(all_true, all_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(all_true, all_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
    "anomaly_accuracy": accuracy_score(all_anomaly_true, all_anomaly_pred),
    "anomaly_f1": f1_score(all_anomaly_true, all_anomaly_pred, zero_division=0),
    "reconstruction_threshold": RECON_THRESHOLD,
}

print("\nOVERALL CROSS-VALIDATION RESULTS")
print(pd.Series(overall_metrics))

print(
    classification_report(
        all_true, all_pred, labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS, zero_division=0,
    )
)

print("Reconstruction threshold:", RECON_THRESHOLD)

pd.DataFrame([overall_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"), index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(all_true, all_pred, labels=np.arange(N_CLASSES)),
    display_labels=FINAL_LABELS,
).plot(values_format="d", xticks_rotation=45)

plt.title("Pure LSTM RNN 3-Head Hybrid Cross-Validation")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()

In [ ]:
# ==========================================================
# CELL 15 — train final model on all sessions
# ==========================================================
(
    X_all,
    y_class_all,
    y_anomaly_all,
    all_files,
    all_starts,
    all_groups,
) = build_dataset_from_sessions(
    sessions
)

normal_all_mask = (
    y_class_all == LABEL_TO_ID["normal"]
)

final_mean, final_variance = calculate_normalization_statistics(
    X_all[normal_all_mask]
)

X_all_normalized = normalize_windows_numpy(
    X_all,
    final_mean,
    final_variance,
)

final_model = build_hybrid_model(
    final_mean,
    final_variance,
)

final_sample_weights = make_training_sample_weights(
    y_class_all,
    y_anomaly_all,
    all_groups,
)

FINAL_EPOCHS = int(
    np.median(cv_best_epochs)
)
FINAL_EPOCHS = max(FINAL_EPOCHS, 1)

print("\n" + "=" * 70)
print("FINAL DEPLOYMENT TRAINING ON 100% OF DATA")
print("=" * 70)
print("Sessions used:", len(np.unique(all_groups)))
print("Windows used:", len(X_all))
print("Final epochs from median CV best epoch:", FINAL_EPOCHS)
print("No validation split is removed from the final training dataset.")

final_history = final_model.fit(
    X_all,
    {
        "reconstruction_output": X_all_normalized,
        "anomaly_output": y_anomaly_all,
        "class_output": y_class_all,
    },
    sample_weight=final_sample_weights,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
    shuffle=True,
)


final_prediction_dict = final_model.predict(
    X_all,
    batch_size=BATCH_SIZE,
    verbose=0,
)

final_class_probabilities = final_prediction_dict[
    "class_output"
]
final_class_predictions = np.argmax(
    final_class_probabilities,
    axis=1,
)

print("\nFINAL TRAINING-DATA CLASSIFICATION REPORT")
print(
    classification_report(
        y_class_all,
        final_class_predictions,
        labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS,
        zero_division=0,
    )
)
print(
    "Final training-data accuracy:",
    accuracy_score(
        y_class_all,
        final_class_predictions,
    ),
)
print(
    "NOTE: this is training-set fit, not an unbiased generalization result. "
    "Use the cross-validation metrics for model performance."
)

pd.DataFrame(final_history.history).to_csv(
    os.path.join(
        FINAL_DIR,
        "final_training_history.csv",
    ),
    index=False,
)

In [ ]:
# ==========================================================
# CELL 16 — save model, labels, metadata, thresholds
# (CHANGED: thresholds dict now includes global_normal_baseline,
# local_normal_baselines, and severity multipliers.)
# ==========================================================
KERAS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "pure_lstm_ae_anomaly_classifier.keras"))
H5_PATH = os.path.abspath(os.path.join(FINAL_DIR, "pure_lstm_ae_anomaly_classifier.h5"))
LABELS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "labels.txt"))
METADATA_PATH = os.path.abspath(os.path.join(FINAL_DIR, "metadata.json"))
THRESHOLDS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "thresholds.json"))

final_model.save(KERAS_PATH)
final_model.save(H5_PATH)

with open(LABELS_PATH, "w", encoding="utf-8") as file:
    file.write("\n".join(FINAL_LABELS))

thresholds = {
    "global_reconstruction_fallback": RECON_THRESHOLD,
    "global_normal_baseline": GLOBAL_NORMAL_BASELINE,
    "state_reconstruction_thresholds": STATE_RECON_THRESHOLDS,
    "pwm_bin_edges": PWM_BIN_EDGES.tolist(),
    "pwm_bin_names": PWM_BIN_NAMES,
    "load_proxy_feature": LOAD_PROXY_FEATURE,
    "global_normal_load_split": GLOBAL_NORMAL_LOAD_SPLIT,
    "pwm_load_splits": PWM_LOAD_SPLITS,
    "normal_pwm_reconstruction_thresholds": NORMAL_PWM_RECON_THRESHOLDS,
    "normal_pwm_local_baselines": LOCAL_NORMAL_BASELINES,
    "normal_pwm_load_reconstruction_thresholds": NORMAL_PWM_LOAD_RECON_THRESHOLDS,
    "anomaly_probability_threshold": ANOMALY_PROB_THRESHOLD,
    "class_confidence_threshold": CLASS_CONF_THRESHOLD,
    "severity_mild_multiplier": SEVERITY_MILD_MULTIPLIER,
    "severity_moderate_multiplier": SEVERITY_MODERATE_MULTIPLIER,
    "severity_severe_multiplier": SEVERITY_SEVERE_MULTIPLIER,
}

metadata = {
    "model_name": "pure_lstm_ae_anomaly_classifier",
    "input_shape": [WINDOW_LEN, N_FEATURES],
    "feature_columns": FEATURE_COLUMNS,
    "labels": FINAL_LABELS,
    "safe_labels": SAFE_LABELS,
    "fault_labels": FAULT_LABELS,
    "scale_divisor": SCALE_DIVISOR,
    "window_step": WINDOW_STEP,
    "row_period_sec": ROW_PERIOD_SEC,
    "parameter_count": int(final_model.count_params()),
    "normalization_mean": final_mean.tolist(),
    "normalization_variance": final_variance.tolist(),
    "final_training_epochs": FINAL_EPOCHS,
    "thresholds": thresholds,
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

with open(THRESHOLDS_PATH, "w", encoding="utf-8") as file:
    json.dump(thresholds, file, indent=2)

print("Saved:", KERAS_PATH)
print("Saved:", H5_PATH)
print("Saved:", LABELS_PATH)
print("Saved:", METADATA_PATH)
print("Saved:", THRESHOLDS_PATH)

In [ ]:
# ==========================================================
# CELL 17 — corrected one-input ONNX export for STM32Cube.AI
# ==========================================================
import os

import numpy as np
import onnx
import onnxruntime as ort
import tensorflow as tf
import tf2onnx


CUBE_ONNX_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_lstm_ae_3head_cube_ai.onnx",
    )
)


# ----------------------------------------------------------
# 1. Create a model containing everything AFTER the original
#    Keras Normalization layer.
#
#    This prevents tf2onnx from converting normalization mean
#    and variance into extra runtime inputs.
# ----------------------------------------------------------
normalization_output = final_model.get_layer(
    "input_normalization"
).output

deployment_core_model = tf.keras.Model(
    inputs=normalization_output,
    outputs={
        "reconstruction_output":
            final_model.get_layer(
                "reconstruction_output"
            ).output,

        "anomaly_output":
            final_model.get_layer(
                "anomaly_output"
            ).output,

        "class_output":
            final_model.get_layer(
                "class_output"
            ).output,
    },
    name="pure_lstm_deployment_core",
)


# Convert values into ordinary Python lists before tracing.
# tf.constant will therefore create real ONNX initializers,
# not additional ONNX inputs.
export_mean_values = (
    np.asarray(
        final_mean,
        dtype=np.float32,
    )
    .reshape(N_FEATURES)
    .tolist()
)

export_variance_values = (
    np.asarray(
        final_variance,
        dtype=np.float32,
    )
    .reshape(N_FEATURES)
    .tolist()
)


CUBE_INPUT_SIGNATURE = [
    tf.TensorSpec(
        shape=(
            1,
            WINDOW_LEN,
            N_FEATURES,
        ),
        dtype=tf.float32,
        name="motor_window",
    )
]


@tf.function(
    input_signature=CUBE_INPUT_SIGNATURE
)
def cube_ai_export_function(motor_window):
    # These become constants inside the ONNX graph.
    normalization_mean = tf.constant(
        export_mean_values,
        dtype=tf.float32,
        name="embedded_normalization_mean",
    )

    normalization_variance = tf.constant(
        export_variance_values,
        dtype=tf.float32,
        name="embedded_normalization_variance",
    )

    normalization_mean = tf.reshape(
        normalization_mean,
        shape=(
            1,
            1,
            N_FEATURES,
        ),
    )

    normalization_variance = tf.reshape(
        normalization_variance,
        shape=(
            1,
            1,
            N_FEATURES,
        ),
    )

    # This is the same calculation performed by the original
    # Keras Normalization layer.
    normalized_motor_window = (
        motor_window
        - normalization_mean
    ) / tf.sqrt(
        normalization_variance
    )

    outputs = deployment_core_model(
        normalized_motor_window,
        training=False,
    )

    reconstruction = tf.identity(
        outputs["reconstruction_output"],
        name="cube_reconstruction_output",
    )

    anomaly = tf.identity(
        outputs["anomaly_output"],
        name="cube_anomaly_output",
    )

    classification = tf.identity(
        outputs["class_output"],
        name="cube_class_output",
    )

    return (
        reconstruction,
        anomaly,
        classification,
    )


# ----------------------------------------------------------
# 2. Export ONNX
# ----------------------------------------------------------
onnx_model, _ = tf2onnx.convert.from_function(
    cube_ai_export_function,
    input_signature=CUBE_INPUT_SIGNATURE,
    opset=13,
    output_path=CUBE_ONNX_PATH,
)


# ----------------------------------------------------------
# 3. Structural ONNX validation
# ----------------------------------------------------------
checked_model = onnx.load(
    CUBE_ONNX_PATH
)

onnx.checker.check_model(
    checked_model
)

print("ONNX structural validation: PASSED")
print("Saved ONNX model:", CUBE_ONNX_PATH)

print(
    "ONNX model size:",
    round(
        os.path.getsize(CUBE_ONNX_PATH)
        / 1024.0,
        2,
    ),
    "KB",
)


# ----------------------------------------------------------
# 4. Open the exported model
# ----------------------------------------------------------
onnx_session = ort.InferenceSession(
    CUBE_ONNX_PATH,
    providers=[
        "CPUExecutionProvider",
    ],
)


onnx_inputs = onnx_session.get_inputs()

print("\nONNX inputs:")

for model_input in onnx_inputs:
    print(
        model_input.name,
        model_input.shape,
        model_input.type,
    )


# The deployment model must have exactly one input.
if len(onnx_inputs) != 1:
    raise RuntimeError(
        "Incorrect ONNX export: the model has "
        f"{len(onnx_inputs)} inputs instead of one. "
        f"Inputs found: {[x.name for x in onnx_inputs]}"
    )


onnx_input_name = onnx_inputs[0].name

if onnx_input_name != "motor_window":
    raise RuntimeError(
        "Unexpected ONNX input name: "
        f"{onnx_input_name}"
    )

print("\nONNX one-input validation: PASSED")


# ----------------------------------------------------------
# 5. Compare Keras and ONNX predictions
# ----------------------------------------------------------
verification_input = np.asarray(
    X_all[:1],
    dtype=np.float32,
)

keras_outputs = final_model.predict(
    verification_input,
    verbose=0,
)

onnx_output_information = (
    onnx_session.get_outputs()
)

onnx_output_values = onnx_session.run(
    None,
    {
        "motor_window": verification_input,
    },
)


onnx_reconstruction = None
onnx_anomaly = None
onnx_classification = None


print("\nONNX outputs:")

for information, value in zip(
    onnx_output_information,
    onnx_output_values,
):
    print(
        information.name,
        value.shape,
        value.dtype,
    )

    if tuple(value.shape) == (
        1,
        WINDOW_LEN,
        N_FEATURES,
    ):
        onnx_reconstruction = value

    elif tuple(value.shape) == (
        1,
        1,
    ):
        onnx_anomaly = value

    elif tuple(value.shape) == (
        1,
        N_CLASSES,
    ):
        onnx_classification = value


if onnx_reconstruction is None:
    raise RuntimeError(
        "Reconstruction output was not found."
    )

if onnx_anomaly is None:
    raise RuntimeError(
        "Anomaly output was not found."
    )

if onnx_classification is None:
    raise RuntimeError(
        "Classification output was not found."
    )


reconstruction_difference = float(
    np.max(
        np.abs(
            np.asarray(
                keras_outputs["reconstruction_output"]
            )
            - onnx_reconstruction
        )
    )
)

anomaly_difference = float(
    np.max(
        np.abs(
            np.asarray(
                keras_outputs["anomaly_output"]
            )
            - onnx_anomaly
        )
    )
)

classification_difference = float(
    np.max(
        np.abs(
            np.asarray(
                keras_outputs["class_output"]
            )
            - onnx_classification
        )
    )
)


print(
    "\nMaximum reconstruction difference:",
    reconstruction_difference,
)

print(
    "Maximum anomaly difference:",
    anomaly_difference,
)

print(
    "Maximum classification difference:",
    classification_difference,
)


MAX_ALLOWED_EXPORT_DIFFERENCE = 1.0e-4

maximum_difference = max(
    reconstruction_difference,
    anomaly_difference,
    classification_difference,
)

if not np.isfinite(maximum_difference):
    raise RuntimeError(
        "ONNX validation produced NaN or infinity."
    )

if maximum_difference > MAX_ALLOWED_EXPORT_DIFFERENCE:
    raise RuntimeError(
        "ONNX predictions differ too much from Keras. "
        f"Maximum difference: {maximum_difference}"
    )


print("\nKeras versus ONNX validation: PASSED")
print("The ONNX model has one input: motor_window")
print("Input shape: 1 x 30 x 14")
print("\nUse this model in STM32Cube.AI:")
print(CUBE_ONNX_PATH)

In [ ]:
# ==========================================================
# CELL 18 — Float32 TFLite export for fixed-length LSTM
# ==========================================================

TFLITE_FLOAT32_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_lstm_only_3head_float32.tflite",
    )
)

# Confirm that every LSTM was created with unroll=True.
incorrect_lstm_layers = [
    layer.name
    for layer in final_model.layers
    if (
        isinstance(layer, tf.keras.layers.LSTM)
        and not layer.unroll
    )
]

if incorrect_lstm_layers:
    raise RuntimeError(
        "These LSTM layers still use unroll=False: "
        + ", ".join(incorrect_lstm_layers)
        + ". Change every LSTM to unroll=True, rebuild the model, "
          "copy/retrain the weights, and run this cell again."
    )

float32_converter = (
    tf.lite.TFLiteConverter.from_keras_model(
        final_model
    )
)

# Use only native TFLite operations.
float32_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

# Keep model and I/O in Float32.
float32_converter.optimizations = []
float32_converter.inference_input_type = tf.float32
float32_converter.inference_output_type = tf.float32

# Do not use Flex / SELECT_TF_OPS.
float32_converter._experimental_lower_tensor_list_ops = True

tflite_float32_model = float32_converter.convert()

with open(
    TFLITE_FLOAT32_PATH,
    "wb",
) as file:
    file.write(tflite_float32_model)

float32_size_kb = (
    os.path.getsize(TFLITE_FLOAT32_PATH)
    / 1024.0
)

print("Saved Float32 TFLite:", TFLITE_FLOAT32_PATH)
print(
    "Float32 TFLite size KB:",
    round(float32_size_kb, 2),
)

# Verify the generated model.
float32_interpreter = tf.lite.Interpreter(
    model_path=TFLITE_FLOAT32_PATH
)

float32_interpreter.allocate_tensors()

print("\nFLOAT32 INPUT")
for detail in float32_interpreter.get_input_details():
    print(
        "name:",
        detail["name"],
        "| shape:",
        detail["shape"],
        "| dtype:",
        detail["dtype"],
    )

print("\nFLOAT32 OUTPUTS")
for detail in float32_interpreter.get_output_details():
    print(
        "name:",
        detail["name"],
        "| shape:",
        detail["shape"],
        "| dtype:",
        detail["dtype"],
    )

In [ ]:
# ==========================================================
# CELL 19 — export complete model as one C header
# (CHANGED: this notebook's deployment path is ONNX (Cell 17), not
# TFLite — Cell 18's Float32 TFLite export is commented out on
# purpose. Point the header at CUBE_ONNX_PATH instead.)
# ==========================================================
MODEL_HEADER_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "pure_lstm_ae_anomaly_classifier_model.h")
)

def write_binary_model_header(binary_model_path, header_path, array_name):
    with open(binary_model_path, "rb") as file:
        model_bytes = file.read()

    include_guard = array_name.upper() + "_H"

    with open(header_path, "w", encoding="utf-8") as file:
        file.write(f"#ifndef {include_guard}\n")
        file.write(f"#define {include_guard}\n\n")
        file.write("#include <stdint.h>\n")
        file.write("#include <stddef.h>\n\n")
        file.write("#if defined(__GNUC__)\n")
        file.write("#define MODEL_ALIGN __attribute__((aligned(16)))\n")
        file.write("#else\n")
        file.write("#define MODEL_ALIGN\n")
        file.write("#endif\n\n")

        file.write(
            f"static const MODEL_ALIGN uint8_t {array_name}"
            f"[{len(model_bytes)}] = {{\n"
        )

        for start in range(0, len(model_bytes), 12):
            chunk = model_bytes[start:start + 12]
            line = ", ".join(f"0x{byte:02X}" for byte in chunk)
            file.write(f"    {line},\n")

        file.write("};\n\n")
        file.write(
            f"static const size_t {array_name}_len = {len(model_bytes)}u;\n\n"
        )
        file.write(f"#endif /* {include_guard} */\n")

    return len(model_bytes)


header_model_size = write_binary_model_header(
    binary_model_path=CUBE_ONNX_PATH,   # from CELL 17, not TFLite
    header_path=MODEL_HEADER_PATH,
    array_name="pure_lstm_ae_anomaly_classifier_model",
)

print("Saved complete model header:", MODEL_HEADER_PATH)
print("Binary model size KB:", round(header_model_size / 1024, 2))

In [ ]:
# ==========================================================
# CELL 20 — export firmware parameters header
# (CHANGED: added severity multiplier defines, global baseline
# define, and a local-baselines array, matching DANN/CNN headers.)
# ==========================================================
PARAMS_HEADER_PATH = os.path.abspath(os.path.join(FINAL_DIR, "motor_model_params.h"))


def c_float(value):
    return f"{float(value):.9g}f"


normal_low_load_thresholds = np.asarray([
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name]["low_load"]
    for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

normal_high_load_thresholds = np.asarray([
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name]["high_load"]
    for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

pwm_load_split_values = np.asarray([
    PWM_LOAD_SPLITS[bin_name] for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

# ADDED
local_normal_baseline_values = np.asarray([
    LOCAL_NORMAL_BASELINES[bin_name] for bin_name in PWM_BIN_NAMES
], dtype=np.float32)


with open(PARAMS_HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef MOTOR_MODEL_PARAMS_H\n")
    file.write("#define MOTOR_MODEL_PARAMS_H\n\n")
    file.write(f"#define MOTOR_MODEL_WINDOW_LEN {WINDOW_LEN}\n")
    file.write(f"#define MOTOR_MODEL_N_FEATURES {N_FEATURES}\n")
    file.write(f"#define MOTOR_MODEL_INPUT_SIZE {WINDOW_LEN * N_FEATURES}\n")
    file.write(f"#define MOTOR_MODEL_N_CLASSES {N_CLASSES}\n")
    file.write(f"#define MOTOR_MODEL_N_PWM_BINS {len(PWM_BIN_NAMES)}\n")
    file.write(f"#define MOTOR_MODEL_PWM_FEATURE_INDEX {FEATURE_COLUMNS.index('PWM')}\n")
    file.write(f"#define MOTOR_MODEL_RECON_OUTPUT_SIZE {WINDOW_LEN * N_FEATURES}\n")
    file.write(
        "#define MOTOR_MODEL_LOAD_PROXY_FEATURE_INDEX "
        f"{FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)}\n"
    )
    file.write(f"#define MOTOR_MODEL_ANOMALY_THRESHOLD {c_float(ANOMALY_PROB_THRESHOLD)}\n")
    file.write(f"#define MOTOR_MODEL_CLASS_CONF_THRESHOLD {c_float(CLASS_CONF_THRESHOLD)}\n")
    file.write(f"#define MOTOR_MODEL_RECON_THRESHOLD {c_float(RECON_THRESHOLD)}\n")
    # ADDED
    file.write(f"#define MOTOR_MODEL_GLOBAL_NORMAL_BASELINE {c_float(GLOBAL_NORMAL_BASELINE)}\n")
    file.write(f"#define MOTOR_MODEL_SEVERITY_MILD_MULTIPLIER {c_float(SEVERITY_MILD_MULTIPLIER)}\n")
    file.write(f"#define MOTOR_MODEL_SEVERITY_MODERATE_MULTIPLIER {c_float(SEVERITY_MODERATE_MULTIPLIER)}\n")
    file.write(f"#define MOTOR_MODEL_SEVERITY_SEVERE_MULTIPLIER {c_float(SEVERITY_SEVERE_MULTIPLIER)}\n\n")

    file.write(f"static const float MOTOR_MODEL_NORMALIZATION_MEAN[{N_FEATURES}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in final_mean))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_NORMALIZATION_VARIANCE[{N_FEATURES}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in final_variance))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_PWM_BIN_EDGES[{len(PWM_BIN_EDGES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in PWM_BIN_EDGES))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_PWM_LOAD_SPLITS[{len(PWM_BIN_NAMES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in pwm_load_split_values))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_NORMAL_LOW_LOAD_THRESHOLDS[{len(PWM_BIN_NAMES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in normal_low_load_thresholds))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_NORMAL_HIGH_LOAD_THRESHOLDS[{len(PWM_BIN_NAMES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in normal_high_load_thresholds))
    file.write("\n};\n\n")

    # ADDED
    file.write(f"static const float MOTOR_MODEL_LOCAL_NORMAL_BASELINES[{len(PWM_BIN_NAMES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in local_normal_baseline_values))
    file.write("\n};\n\n")

    file.write(f"#define MOTOR_MODEL_MOTOR_OFF_RECON_THRESHOLD {c_float(STATE_RECON_THRESHOLDS['motor_off'])}\n")
    file.write(f"#define MOTOR_MODEL_TURNING_OFF_RECON_THRESHOLD {c_float(STATE_RECON_THRESHOLDS['turning_off'])}\n")
    file.write(f"#define MOTOR_MODEL_TURNING_ON_RECON_THRESHOLD {c_float(STATE_RECON_THRESHOLDS['turning_on'])}\n\n")

    file.write(f"static const char *const MOTOR_MODEL_LABELS[{N_CLASSES}] = {{\n")
    for label in FINAL_LABELS:
        file.write(f'    "{label}",\n')
    file.write("};\n\n")
    file.write("#endif /* MOTOR_MODEL_PARAMS_H */\n")

print("Saved firmware parameters header:", PARAMS_HEADER_PATH)

In [ ]:
# # ==========================================================
# # CELL 21 — verify TFLite model interfaces
# # ==========================================================
# interpreter = tf.lite.Interpreter(
#     model_path=TFLITE_FLOAT32_PATH
# )

# interpreter.allocate_tensors()

# print("TFLite input details:")

# for detail in interpreter.get_input_details():
#     print(detail)

# print("TFLite output details:")

# for detail in interpreter.get_output_details():
#     print(detail)

In [ ]:
# ==========================================================
# CELL 22 — manual classification test
# (CHANGED: includes severity_ratio/severity_label/recommended_action
# in the output + a 5th chart plotting severity over time.)
# ==========================================================
def manual_classification_prediction(test_file):
    model = tf.keras.models.load_model(KERAS_PATH, compile=False)

    test_df = read_one(test_file)

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32)
    )

    if X_test is None or len(X_test) == 0:
        raise RuntimeError(
            "The selected file does not contain enough valid rows for one 30-row window."
        )

    X_test_normalized = normalize_windows_numpy(X_test, final_mean, final_variance)

    prediction_dict = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    reconstructed = prediction_dict["reconstruction_output"]
    anomaly_probabilities = prediction_dict["anomaly_output"].reshape(-1)
    class_probabilities = prediction_dict["class_output"]

    reconstruction_mse = np.mean(
        np.square(reconstructed - X_test_normalized), axis=(1, 2),
    )

    pwm_feature_index = FEATURE_COLUMNS.index("PWM")
    irms_feature_index = FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)

    window_mean_pwm = np.mean(X_test[:, :, pwm_feature_index], axis=1).astype(np.float32)
    window_mean_irms = np.mean(X_test[:, :, irms_feature_index], axis=1).astype(np.float32)

    preliminary_class_predictions = np.argmax(class_probabilities, axis=1).astype(np.int64)

    window_reconstruction_thresholds = np.asarray([
        get_conditional_reconstruction_threshold(class_id, pwm, irms)
        for class_id, pwm, irms in zip(
            preliminary_class_predictions, window_mean_pwm, window_mean_irms,
        )
    ], dtype=np.float32)

    (
        class_predictions,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        anomaly_flags,
        final_decisions,
        local_baselines,
        severity_ratios,
        severity_labels,
        recommended_actions,
    ) = fuse_decisions(
        class_probabilities,
        anomaly_probabilities,
        reconstruction_mse,
        window_reconstruction_thresholds,
        window_mean_pwm,
    )

    normal_running_flags = (
        class_predictions == LABEL_TO_ID["normal"]
    )

    early_warning_candidates = (
        normal_running_flags
        & reconstruction_anomaly_flags
    )

    early_warning_streak = np.zeros(
        len(early_warning_candidates),
        dtype=np.int32,
    )

    for index, candidate in enumerate(early_warning_candidates):
        if candidate:
            early_warning_streak[index] = (
                early_warning_streak[index - 1] + 1
                if index > 0 else 1
            )

    early_warning_flags = (
        early_warning_streak
        >= EARLY_WARNING_CONSECUTIVE_WINDOWS
    )    

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "mean_pwm": window_mean_pwm,
        "mean_irms": window_mean_irms,
        "pred_class_id": class_predictions,
        "pred_label": [FINAL_LABELS[i] for i in class_predictions],
        "class_confidence": class_confidence,
        "anomaly_probability": anomaly_probabilities,
        "binary_anomaly_flag": binary_anomaly_flags.astype(np.int64),
        "reconstruction_mse": reconstruction_mse,
        "reconstruction_threshold": window_reconstruction_thresholds,
        "local_baseline": local_baselines,
        "severity_ratio": severity_ratios,
        "severity_label": severity_labels,
        "recommended_action": recommended_actions,
        "reconstruction_anomaly_flag": reconstruction_anomaly_flags.astype(np.int64),
        "anomaly_flag": anomaly_flags.astype(np.int64),
        "final_decision": final_decisions,
        "early_warning_candidate": early_warning_candidates.astype(np.int64),
        "early_warning_streak": early_warning_streak,
        "early_warning_flag": early_warning_flags.astype(np.int64),
    })

    for class_id, label in enumerate(FINAL_LABELS):
        result[f"probability_{label}"] = class_probabilities[:, class_id]

    majority_class_id = int(result["pred_class_id"].value_counts().idxmax())
    majority_label = FINAL_LABELS[majority_class_id]
    majority_final_decision = result["final_decision"].value_counts().idxmax()
    majority_severity = result["severity_label"].value_counts().idxmax()

    stem = Path(test_file).stem
    output_csv = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}.csv")
    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 70)
    print("MANUAL PURE LSTM RNN 3-HEAD TEST")
    print("=" * 70)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Majority class:", majority_label)
    print("Majority final decision:", majority_final_decision)
    print("Majority severity tier:", majority_severity)
    print("Mean class confidence:", float(np.mean(class_confidence)))
    print("Mean anomaly probability:", float(np.mean(anomaly_probabilities)))
    print("Mean reconstruction MSE:", float(np.mean(reconstruction_mse)))
    print("Anomalous windows:", int(np.sum(anomaly_flags)), "/", len(result))
    print("Early-warning windows:", int(np.sum(early_warning_flags)), "/",len(result),)

    if np.any(early_warning_flags):
        first_warning_time = float(
            result.loc[
                result["early_warning_flag"] == 1,
                "time_sec",
            ].iloc[0]
        )
        print("First EARLY_WARNING at:", first_warning_time, "s")
    else:
        print("No EARLY_WARNING generated.")
        
    print("Saved CSV:", output_csv)

    print("\nFinal decisions:")
    print(result["final_decision"].value_counts())

    print("\nSeverity tiers:")
    print(result["severity_label"].value_counts())

    # CHART 1 — multi-class classification head
    plt.figure(figsize=(14, 7))
    for class_id, label in enumerate(FINAL_LABELS):
        plt.plot(result["time_sec"], class_probabilities[:, class_id], label=label, linewidth=1.5)
    plt.axhline(y=CLASS_CONF_THRESHOLD, linestyle="--", label=f"Confidence threshold ({CLASS_CONF_THRESHOLD:.2f})")
    plt.xlabel("Time [s]"); plt.ylabel("Class probability"); plt.ylim(0.0, 1.05)
    plt.title(f"Classification head probabilities — {stem}")
    plt.legend(loc="upper right", ncol=2); plt.grid(True, alpha=0.3); plt.tight_layout()
    class_head_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_class_head.png")
    plt.savefig(class_head_path, dpi=180); plt.show(); plt.close()
    print("Saved classification-head chart:", class_head_path)

    # CHART 2 — binary anomaly head
    plt.figure(figsize=(14, 5))
    plt.plot(result["time_sec"], anomaly_probabilities, label="Anomaly probability", linewidth=2)
    plt.axhline(y=ANOMALY_PROB_THRESHOLD, linestyle="--", label=f"Anomaly threshold ({ANOMALY_PROB_THRESHOLD:.2f})")
    plt.fill_between(result["time_sec"], 0, 1, where=(anomaly_probabilities >= ANOMALY_PROB_THRESHOLD), alpha=0.15, label="Anomaly detected")
    plt.xlabel("Time [s]"); plt.ylabel("Anomaly probability"); plt.ylim(0.0, 1.05)
    plt.title(f"Binary anomaly head — {stem}")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    anomaly_head_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_anomaly_head.png")
    plt.savefig(anomaly_head_path, dpi=180); plt.show(); plt.close()
    print("Saved anomaly-head chart:", anomaly_head_path)

    # CHART 3 — reconstruction autoencoder head
    plt.figure(figsize=(14, 5))
    plt.plot(result["time_sec"], reconstruction_mse, label="Reconstruction MSE", linewidth=2)
    plt.plot(result["time_sec"], window_reconstruction_thresholds, linestyle="--", linewidth=2, label="State/PWM/load-conditioned threshold")
    plt.fill_between(result["time_sec"], 0, reconstruction_mse, where=(reconstruction_mse > window_reconstruction_thresholds), alpha=0.15, label="Reconstruction anomaly")
    plt.xlabel("Time [s]"); plt.ylabel("Reconstruction MSE")
    plt.title(f"Autoencoder reconstruction head — {stem}")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    reconstruction_head_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_reconstruction_head.png")
    plt.savefig(reconstruction_head_path, dpi=180); plt.show(); plt.close()
    print("Saved reconstruction-head chart:", reconstruction_head_path)

    # CHART 4 — predicted class timeline
    plt.figure(figsize=(14, 5))
    plt.step(result["time_sec"], result["pred_class_id"], where="mid", linewidth=2)
    plt.yticks(np.arange(N_CLASSES), FINAL_LABELS)
    plt.xlabel("Time [s]"); plt.ylabel("Predicted class")
    plt.title(f"Classification result timeline — {stem}")
    plt.grid(True, alpha=0.3); plt.tight_layout()
    timeline_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_class_timeline.png")
    plt.savefig(timeline_path, dpi=180); plt.show(); plt.close()
    print("Saved class timeline:", timeline_path)

    # CHART 5 (ADDED) — severity ratio over time
    plt.figure(figsize=(14, 5))
    plt.plot(result["time_sec"], result["severity_ratio"], linewidth=2, color="darkorange", label="Severity ratio (MSE / local baseline)")
    for level, color in [
        (SEVERITY_MILD_MULTIPLIER, "gold"),
        (SEVERITY_MODERATE_MULTIPLIER, "orange"),
        (SEVERITY_SEVERE_MULTIPLIER, "red"),
    ]:
        plt.axhline(y=level, linestyle="--", color=color, alpha=0.7)
    plt.xlabel("Time [s]"); plt.ylabel("Severity ratio")
    plt.title(f"Reconstruction severity — {stem}")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    severity_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_severity.png")
    plt.savefig(severity_path, dpi=180); plt.show(); plt.close()
    print("Saved severity chart:", severity_path)

    return result

In [ ]:
# ==========================================================
# CELL 23 — select manual test file
# ==========================================================
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

TEST_FILE = filedialog.askopenfilename(
    title="Select motor CSV or Excel file",
    filetypes=[
        ("Excel files", "*.xlsx"),
        ("CSV files", "*.csv"),
        ("All files", "*.*"),
    ],
)

root.destroy()

if TEST_FILE:
    manual_result = manual_classification_prediction(
        TEST_FILE
    )
else:
    print("No file selected.")


In [ ]:
# ==========================================================
# CELL 24 — STANDALONE MANUAL TEST
# Run ONLY this cell. No training or previous cells needed.
# Uses saved model, metadata and calibrated thresholds.
# ==========================================================

import os
import json
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


# ==========================================================
# 1. FIND SAVED MODEL
# ==========================================================

OUTPUT_ROOT = Path.cwd() / "lstm_rnn_ae_3head_output"
MODEL_DIR = OUTPUT_ROOT / "final_model"

MODEL_PATH = (
    MODEL_DIR / "pure_lstm_ae_anomaly_classifier.keras"
)

METADATA_PATH = MODEL_DIR / "metadata.json"
THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"


def select_file(title, filetypes, initialdir=None):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    path = filedialog.askopenfilename(
        title=title,
        initialdir=str(initialdir or Path.cwd()),
        filetypes=filetypes
    )

    root.destroy()
    return Path(path) if path else None


# If model is not found automatically, select it once.
if not MODEL_PATH.is_file():

    print("Saved model not found in default location.")

    MODEL_PATH = select_file(
        "Select your trained LSTM .keras model",
        [("Keras model", "*.keras")]
    )

    if MODEL_PATH is None:
        raise RuntimeError("No model selected.")

    MODEL_DIR = MODEL_PATH.parent
    OUTPUT_ROOT = MODEL_DIR.parent

    METADATA_PATH = MODEL_DIR / "metadata.json"
    THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"


if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"metadata.json not found: {METADATA_PATH}"
    )

if not THRESHOLDS_PATH.is_file():
    raise FileNotFoundError(
        f"thresholds.json not found: {THRESHOLDS_PATH}"
    )


# ==========================================================
# 2. LOAD SAVED CONFIGURATION AND MODEL
# ==========================================================

with open(METADATA_PATH, encoding="utf-8") as f:
    meta = json.load(f)

with open(THRESHOLDS_PATH, encoding="utf-8") as f:
    thresholds = json.load(f)

FEATURES = meta["feature_columns"]
LABELS = meta["labels"]

WIN, NF = meta["input_shape"]
STEP = int(meta["window_step"])
ROW_SEC = float(meta["row_period_sec"])
SCALE = float(meta["scale_divisor"])

MEAN = np.asarray(
    meta["normalization_mean"], dtype=np.float32
)

VAR = np.asarray(
    meta["normalization_variance"], dtype=np.float32
)

SAFE_LABELS = meta["safe_labels"]
FAULT_LABELS = meta["fault_labels"]

ANOM_THR = float(
    thresholds["anomaly_probability_threshold"]
)

CLASS_THR = float(
    thresholds["class_confidence_threshold"]
)

GLOBAL_RECON_THR = float(
    thresholds["global_reconstruction_fallback"]
)

GLOBAL_BASELINE = float(
    thresholds["global_normal_baseline"]
)

STATE_THRESHOLDS = thresholds[
    "state_reconstruction_thresholds"
]

PWM_EDGES = np.asarray(
    thresholds["pwm_bin_edges"], dtype=np.float32
)

PWM_NAMES = thresholds["pwm_bin_names"]

PWM_SPLITS = thresholds["pwm_load_splits"]

NORMAL_PWM_THRESHOLDS = thresholds[
    "normal_pwm_reconstruction_thresholds"
]

NORMAL_LOAD_THRESHOLDS = thresholds[
    "normal_pwm_load_reconstruction_thresholds"
]

LOCAL_BASELINES = thresholds[
    "normal_pwm_local_baselines"
]

GLOBAL_LOAD_SPLIT = float(
    thresholds["global_normal_load_split"]
)

MILD = float(thresholds["severity_mild_multiplier"])
MODERATE = float(thresholds["severity_moderate_multiplier"])
SEVERE = float(thresholds["severity_severe_multiplier"])

EARLY_WARNING_WINDOWS = 3
BATCH_SIZE = 128


print("Loading trained LSTM model...")

model = tf.keras.models.load_model(
    str(MODEL_PATH),
    compile=False
)

print("Model loaded successfully.")
print("Model:", MODEL_PATH)
print("Input:", WIN, "x", NF)


# ==========================================================
# 3. SELECT MANUAL TEST FILE
# ==========================================================

TEST_FILE = select_file(
    "Select motor CSV or Excel file for manual testing",
    [
        ("Motor data files", "*.xlsx *.csv"),
        ("Excel files", "*.xlsx"),
        ("CSV files", "*.csv"),
        ("All files", "*.*")
    ]
)

if TEST_FILE is None:
    raise RuntimeError("Manual test cancelled.")


# ==========================================================
# 4. CREATE RESULT DIRECTORY
# ==========================================================

RESULT_DIR = OUTPUT_ROOT / "manual_test_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

stem = TEST_FILE.stem

print("\nSelected file:", TEST_FILE)
print("Results directory:", RESULT_DIR)


# ==========================================================
# 5. LOAD DATA — SAME PREPROCESSING AS TRAINING
# ==========================================================

if TEST_FILE.suffix.lower() == ".xlsx":
    df = pd.read_excel(TEST_FILE)

elif TEST_FILE.suffix.lower() == ".csv":
    df = pd.read_csv(TEST_FILE)

else:
    raise ValueError("Only CSV and XLSX files supported.")


missing = [c for c in FEATURES if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required feature columns: {missing}"
    )


data = df[FEATURES].apply(
    pd.to_numeric, errors="coerce"
)

data = data / SCALE

# Same PWM clipping as training Cell 4.
data["PWM"] = data["PWM"].clip(upper=1.0)

data = data.replace(
    [np.inf, -np.inf], np.nan
).dropna().reset_index(drop=True)

X = data.to_numpy(dtype=np.float32)

if len(X) < WIN:
    raise ValueError(
        f"Only {len(X)} valid rows. Minimum {WIN} required."
    )


# ==========================================================
# 6. CREATE WINDOWS
# ==========================================================

starts = np.arange(
    0, len(X) - WIN + 1, STEP, dtype=np.int64
)

windows = np.stack(
    [X[s:s + WIN] for s in starts]
).astype(np.float32)

print("Valid rows:", len(X))
print("Total windows:", len(windows))


# ==========================================================
# 7. RUN MODEL INFERENCE
# ==========================================================

predictions = model.predict(
    windows,
    batch_size=BATCH_SIZE,
    verbose=0
)

# Handle both dictionary and list Keras outputs.
if isinstance(predictions, dict):

    reconstructed = predictions["reconstruction_output"]

    anomaly_probability = predictions[
        "anomaly_output"
    ].reshape(-1)

    class_probability = predictions["class_output"]

else:

    mapped = dict(zip(model.output_names, predictions))

    reconstructed = mapped["reconstruction_output"]

    anomaly_probability = mapped[
        "anomaly_output"
    ].reshape(-1)

    class_probability = mapped["class_output"]


# ==========================================================
# 8. RECONSTRUCTION MSE — SAME AS ORIGINAL CELL 22
# ==========================================================

normalized = (
    windows - MEAN.reshape(1, 1, NF)
) / np.sqrt(
    VAR.reshape(1, 1, NF) + 1e-7
)

reconstruction_mse = np.mean(
    np.square(reconstructed - normalized),
    axis=(1, 2)
)

PWM_INDEX = FEATURES.index("PWM")
IRMS_INDEX = FEATURES.index(
    thresholds["load_proxy_feature"]
)

mean_pwm = windows[:, :, PWM_INDEX].mean(axis=1)
mean_irms = windows[:, :, IRMS_INDEX].mean(axis=1)

class_id = np.argmax(
    class_probability, axis=1
).astype(np.int64)

class_confidence = np.max(
    class_probability, axis=1
)


# ==========================================================
# 9. CONDITIONAL AE THRESHOLD
# ==========================================================

def pwm_bin_index(pwm):
    return int(np.clip(
        np.searchsorted(PWM_EDGES, pwm, side="right") - 1,
        0,
        len(PWM_NAMES) - 1
    ))


def reconstruction_threshold(cid, pwm, irms):

    label = LABELS[int(cid)]

    if label in ["motor_off", "turning_off", "turning_on"]:
        return float(
            STATE_THRESHOLDS.get(
                label, GLOBAL_RECON_THR
            )
        )

    bin_name = PWM_NAMES[pwm_bin_index(pwm)]

    load_split = float(
        PWM_SPLITS.get(bin_name, GLOBAL_LOAD_SPLIT)
    )

    load_state = (
        "high_load" if irms >= load_split
        else "low_load"
    )

    return float(
        NORMAL_LOAD_THRESHOLDS.get(
            bin_name, {}
        ).get(
            load_state,
            NORMAL_PWM_THRESHOLDS.get(
                bin_name, GLOBAL_RECON_THR
            )
        )
    )


recon_thresholds = np.asarray([
    reconstruction_threshold(cid, pwm, irms)
    for cid, pwm, irms in zip(
        class_id, mean_pwm, mean_irms
    )
], dtype=np.float32)


# ==========================================================
# 10. THREE-HEAD DECISION FUSION
# ==========================================================

binary_flags = anomaly_probability >= ANOM_THR

reconstruction_flags = (
    reconstruction_mse > recon_thresholds
)

known_fault_flags = (
    np.isin(
        [LABELS[int(i)] for i in class_id],
        FAULT_LABELS
    )
    & (class_confidence >= CLASS_THR)
)

anomaly_flags = (
    binary_flags |
    reconstruction_flags |
    known_fault_flags
)

final_decisions = []

for i in range(len(class_id)):

    label = LABELS[int(class_id[i])]

    if known_fault_flags[i]:
        decision = label

    elif binary_flags[i] or reconstruction_flags[i]:
        decision = "unknown_fault"

    elif (
        label in SAFE_LABELS
        and class_confidence[i] >= CLASS_THR
    ):
        decision = label

    else:
        decision = "uncertain_safe"

    final_decisions.append(decision)


# ==========================================================
# 11. SEVERITY ESTIMATION
# ==========================================================

local_baselines = np.asarray([
    float(
        LOCAL_BASELINES.get(
            PWM_NAMES[pwm_bin_index(pwm)],
            GLOBAL_BASELINE
        )
    )
    for pwm in mean_pwm
], dtype=np.float32)

safe_baselines = np.where(
    local_baselines > 0,
    local_baselines,
    1.0
)

severity_ratio = reconstruction_mse / safe_baselines

severity_labels = []
recommended_actions = []

for ratio in severity_ratio:

    if ratio < MILD:
        severity_labels.append("normal")
        recommended_actions.append("No action needed.")

    elif ratio < MODERATE:
        severity_labels.append("mild")
        recommended_actions.append(
            "Log and monitor; no immediate action."
        )

    elif ratio < SEVERE:
        severity_labels.append("moderate")
        recommended_actions.append(
            "Schedule inspection soon."
        )

    else:
        severity_labels.append("severe")
        recommended_actions.append(
            "Stop motor / inspect immediately."
        )


# ==========================================================
# 12. EARLY WARNING
# ==========================================================

early_candidates = (
    (class_id == LABELS.index("normal"))
    & reconstruction_flags
)

early_streak = np.zeros(
    len(early_candidates), dtype=np.int32
)

for i, candidate in enumerate(early_candidates):

    if candidate:
        early_streak[i] = (
            early_streak[i - 1] + 1 if i > 0 else 1
        )

early_flags = (
    early_streak >= EARLY_WARNING_WINDOWS
)


# ==========================================================
# 13. CREATE DETAILED RESULTS CSV
# ==========================================================

result = pd.DataFrame({

    "window": np.arange(len(windows)),

    "start_row": starts,

    "time_sec": (
        starts + WIN / 2.0
    ) * ROW_SEC,

    "mean_pwm": mean_pwm,

    "mean_irms": mean_irms,

    "pred_class_id": class_id,

    "pred_label": [
        LABELS[int(i)] for i in class_id
    ],

    "class_confidence": class_confidence,

    "anomaly_probability": anomaly_probability,

    "binary_anomaly_flag":
        binary_flags.astype(int),

    "reconstruction_mse": reconstruction_mse,

    "reconstruction_threshold": recon_thresholds,

    "local_baseline": local_baselines,

    "severity_ratio": severity_ratio,

    "severity_label": severity_labels,

    "recommended_action": recommended_actions,

    "reconstruction_anomaly_flag":
        reconstruction_flags.astype(int),

    "anomaly_flag": anomaly_flags.astype(int),

    "final_decision": final_decisions,

    "early_warning_candidate":
        early_candidates.astype(int),

    "early_warning_streak": early_streak,

    "early_warning_flag":
        early_flags.astype(int)
})

for i, label in enumerate(LABELS):

    result[f"probability_{label}"] = (
        class_probability[:, i]
    )


CSV_PATH = RESULT_DIR / f"manual_test_{stem}.csv"

result.to_csv(CSV_PATH, index=False)


# ==========================================================
# 14. DISPLAY SUMMARY
# ==========================================================

print("\n" + "=" * 70)
print("MANUAL PURE LSTM RNN 3-HEAD TEST")
print("=" * 70)

print("File:", TEST_FILE.name)
print("Total windows:", len(result))

print(
    "Majority class:",
    result["pred_label"].value_counts().idxmax()
)

print(
    "Majority final decision:",
    result["final_decision"].value_counts().idxmax()
)

print(
    "Majority severity:",
    result["severity_label"].value_counts().idxmax()
)

print(
    "Mean class confidence:",
    float(np.mean(class_confidence))
)

print(
    "Mean anomaly probability:",
    float(np.mean(anomaly_probability))
)

print(
    "Mean reconstruction MSE:",
    float(np.mean(reconstruction_mse))
)

print(
    "Anomalous windows:",
    int(anomaly_flags.sum()), "/", len(result)
)

print(
    "Early-warning windows:",
    int(early_flags.sum()), "/", len(result)
)

if early_flags.any():

    first_time = result.loc[
        result["early_warning_flag"] == 1,
        "time_sec"
    ].iloc[0]

    print("First EARLY_WARNING:", first_time, "s")

else:
    print("No EARLY_WARNING generated.")

print("\nFinal decisions:")
print(result["final_decision"].value_counts())

print("\nSeverity tiers:")
print(result["severity_label"].value_counts())


# ==========================================================
# 15. SAVE FIVE ORIGINAL MANUAL-TEST CHARTS
# ==========================================================

time = result["time_sec"].to_numpy()


def save_chart(suffix):
    path = RESULT_DIR / f"manual_test_{stem}_{suffix}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.show()
    plt.close()
    print("Saved:", path)


# CHART 1 — Classification probabilities

plt.figure(figsize=(14, 7))

for i, label in enumerate(LABELS):
    plt.plot(
        time,
        class_probability[:, i],
        label=label,
        linewidth=1.5
    )

plt.axhline(
    CLASS_THR,
    linestyle="--",
    label=f"Confidence threshold ({CLASS_THR:.2f})"
)

plt.xlabel("Time [s]")
plt.ylabel("Class probability")
plt.ylim(0, 1.05)
plt.title(f"Classification head probabilities — {stem}")
plt.legend(loc="upper right", ncol=2)
plt.grid(True, alpha=0.3)

save_chart("class_head")


# CHART 2 — Binary anomaly probability

plt.figure(figsize=(14, 5))

plt.plot(
    time,
    anomaly_probability,
    label="Anomaly probability",
    linewidth=2
)

plt.axhline(
    ANOM_THR,
    linestyle="--",
    label=f"Anomaly threshold ({ANOM_THR:.2f})"
)

plt.fill_between(
    time, 0, 1,
    where=binary_flags,
    alpha=0.15,
    label="Anomaly detected"
)

plt.xlabel("Time [s]")
plt.ylabel("Anomaly probability")
plt.ylim(0, 1.05)
plt.title(f"Binary anomaly head — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)

save_chart("anomaly_head")


# CHART 3 — Reconstruction MSE

plt.figure(figsize=(14, 5))

plt.plot(
    time,
    reconstruction_mse,
    label="Reconstruction MSE",
    linewidth=2
)

plt.plot(
    time,
    recon_thresholds,
    linestyle="--",
    linewidth=2,
    label="State/PWM/load-conditioned threshold"
)

plt.fill_between(
    time,
    0,
    reconstruction_mse,
    where=reconstruction_flags,
    alpha=0.15,
    label="Reconstruction anomaly"
)

plt.xlabel("Time [s]")
plt.ylabel("Reconstruction MSE")
plt.title(f"Autoencoder reconstruction head — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)

save_chart("reconstruction_head")


# CHART 4 — Predicted class timeline

plt.figure(figsize=(14, 5))

plt.step(
    time,
    class_id,
    where="mid",
    linewidth=2
)

plt.yticks(np.arange(len(LABELS)), LABELS)
plt.xlabel("Time [s]")
plt.ylabel("Predicted class")
plt.title(f"Classification result timeline — {stem}")
plt.grid(True, alpha=0.3)

save_chart("class_timeline")


# CHART 5 — Reconstruction severity

plt.figure(figsize=(14, 5))

plt.plot(
    time,
    severity_ratio,
    linewidth=2,
    color="darkorange",
    label="Severity ratio (MSE / local baseline)"
)

for level, color in [
    (MILD, "gold"),
    (MODERATE, "orange"),
    (SEVERE, "red")
]:
    plt.axhline(
        level,
        linestyle="--",
        color=color,
        alpha=0.7
    )

plt.xlabel("Time [s]")
plt.ylabel("Severity ratio")
plt.title(f"Reconstruction severity — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)

save_chart("severity")


# ==========================================================
# 16. COMPLETE
# ==========================================================

manual_result = result

print("\n" + "=" * 70)
print("MANUAL TEST COMPLETED SUCCESSFULLY")
print("=" * 70)
print("Model:", MODEL_PATH.name)
print("Test file:", TEST_FILE.name)
print("CSV:", CSV_PATH)
print("Five charts saved in:", RESULT_DIR)
print("No model training performed.")